# Lab 09: BERT Fine-tuning

**Module 09** | Read `notes/09-bert.pdf` before running this notebook.

Fine-tune DistilBERT on SST-2 sentiment with the Hugging Face Trainer API and report validation accuracy.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Lab 09: DistilBERT fine-tuning on SST-2

**Sentiment classification** means labeling text as positive or negative. Instead of training a language model from scratch, we **fine-tune** a pre-trained **DistilBERT** model that already understands English. We teach it our specific task using labeled movie reviews (SST-2).


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **Tokenization** | Splitting text into subword pieces and converting each piece to an integer ID the model understands. |
| **Tokenizer** | A Hugging Face tool that encodes text to `input_ids` and builds the `attention_mask`. |
| **Input IDs** | A sequence of integers, one per token, fed into the embedding layer. |
| **Attention mask** | A list of 1s and 0s marking real tokens (1) vs padding tokens (0) so the model ignores padding. |
| **Fine-tuning** | Starting from pre-trained weights and continuing training on a new labeled dataset for a specific task. |
| **Pre-trained model** | A model already trained on large text corpora. It knows grammar and word meaning before your lab data. |
| **Classification head** | A small output layer on top of the encoder that produces class scores (here: negative vs positive). |
| **Validation accuracy** | Fraction of validation examples where the predicted label matches the true label. Higher is better. |
| **Trainer** | Hugging Face helper that runs the training loop, evaluation, logging, and device placement for you. |
| **Logits** | Raw class scores before softmax. The highest logit wins the prediction. |
| **SST-2** | Stanford Sentiment Treebank, binary movie review labels: 0 = negative, 1 = positive. |
| **Epoch** | One complete pass through the training set. We use 1 epoch for a quick classroom demo. |

Refer back to this table whenever a term appears in code or output.


### Concept: Tokenization

Computers do not read words directly. **Tokenization** breaks text into **tokens** (often subword pieces). Example: "loved" might become `["love", "##d"]` because the vocabulary stores common roots and suffix pieces separately.

The **tokenizer** returns:
- `input_ids`: integer IDs for each token position.
- `attention_mask`: 1 where a real token sits, 0 on padding added to reach `max_length`.

Padding makes every sequence the same length so batches stack into tensors. The attention mask tells the model: "ignore padding positions; they carry no information." 


### Concept: Fine-tuning and validation accuracy

**Fine-tuning** updates *all* (or most) weights of a pre-trained encoder using your labeled examples. The model keeps general English knowledge but adjusts its internal representations for sentiment.

**Validation accuracy** is measured on held-out reviews the model did not train on. It estimates real-world performance. Random guessing on binary SST-2 would score 50%; a fine-tuned DistilBERT often exceeds 85% even after one epoch.

**What Trainer does:** Wraps the repetitive training boilerplate: batching, forward pass, loss, backward pass, optimizer step, periodic evaluation, metric logging, and moving tensors to GPU if available. You supply the model, datasets, and `TrainingArguments`; Trainer runs the loop.


### Step 1: Load raw SST-2 examples

**What this does:** Configures the runtime, loads SST-2, and prints split sizes plus three raw sentences with labels.

**Why we do this:** Inspect real text before tokenization so you know what the model will eventually classify.


**What to look for in the output**

Splits train and validation. Train roughly 7,000-9,000 examples (SetFit/sst2 size). Labels 0/1 with readable sentiment names. Text should look like short movie review sentences.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "distilbert-base-uncased"  # smaller, faster cousin of BERT
MAX_LENGTH = 128  # pad/truncate all sequences to this many tokens
BATCH_SIZE = 8
LABEL_NAMES = {0: "negative", 1: "positive"}

# Load Stanford Sentiment Treebank binary classification set
raw = load_dataset("SetFit/sst2")
print("Dataset splits:", list(raw.keys()))
print(f"  Train:      {len(raw['train']):,} examples")
print(f"  Validation: {len(raw['validation']):,} examples")

print("\nFirst 3 raw training examples")
for i in range(3):
    row = raw["train"][i]
    print(f"  [{i}] label={row['label']} ({LABEL_NAMES[row['label']]})")
    print(f"      text: {row['text'][:120]}{'...' if len(row['text']) > 120 else ''}")


### Step 2: Tokenize and decode one example

**What this does:** Downloads the tokenizer, encodes one review, shows tokens and attention mask behavior, then tokenizes the full dataset.

**Why we do this:** Tokenization is the bridge between human-readable strings and model tensors. Decoding confirms subword splitting.


**What to look for in the output**

Active token count well below 128 for short sentences. Tokens list shows word pieces. Decoded text should match the original closely. Encoded columns include input_ids, attention_mask, labels.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example_text = raw["train"][0]["text"]
encoded_one = tokenizer(
    example_text,
    truncation=True,  # cut long reviews at max_length
    max_length=MAX_LENGTH,
    padding="max_length",  # pad short reviews with special padding tokens
)

input_ids = encoded_one["input_ids"]
attention_mask = encoded_one["attention_mask"]
# Keep only positions where mask == 1 (real tokens, not padding)
active_ids = [tid for tid, mask in zip(input_ids, attention_mask) if mask == 1]
tokens = tokenizer.convert_ids_to_tokens(active_ids)

print("Tokenization demo (training example 0)")
print(f"  Text: {example_text}")
print(f"  Label: {raw['train'][0]['label']} ({LABEL_NAMES[raw['train'][0]['label']]})")
print(f"  Active token count: {len(active_ids)}")
print(f"  Tokens: {tokens}")
print(f"  Decoded: {tokenizer.decode(active_ids)}")

def tokenize_batch(batch: dict) -> dict:
    # Applied to every row when we call raw.map(...)
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

encoded = raw.map(tokenize_batch, batched=True)
encoded = encoded.rename_column("label", "labels")  # Trainer expects 'labels'
encoded.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print(f"\nEncoded columns: {encoded['train'].column_names}")


### Step 3: Load model and count trainable parameters

**What this does:** Loads DistilBERT with a 2-class head, moves it to `device`, and prints parameter counts.

**Why we do this:** You see how large the model is and confirm every weight will be fine-tuned (nothing frozen in this demo).


**What to look for in the output**

Total parameters around 66-67 million for DistilBERT-base. Trainable equals total (0 frozen). Device matches your runtime cell (cpu or cuda).


In [ ]:
# num_labels=2 adds a classification head for negative vs positive
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model summary")
print(f"  Model:             {MODEL_NAME}")
print(f"  Device:            {device}")
print(f"  Total parameters:  {total_params:,}")
print(f"  Trainable params:  {trainable_params:,}")
print(f"  Frozen params:     {total_params - trainable_params:,}")


### Step 4: Train for one epoch and evaluate

**What this does:** Defines a metric function, configures `TrainingArguments`, runs `Trainer.train()` and `Trainer.evaluate()`.

**Why we do this:** One epoch is enough to see accuracy rise above chance. Trainer handles the optimization loop so you can focus on data and metrics.


**What to look for in the output**

Training loss printed (often 0.2-0.5 after one epoch). Validation accuracy typically 0.85+ (85%+). Validation loss lower than random baseline.


In [ ]:
def compute_metrics(eval_pred):
    # eval_pred = (logits, labels) tuple from Trainer
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)  # pick class with highest score
    accuracy = (preds == labels).mean()
    return {"accuracy": float(accuracy)}


training_args = TrainingArguments(
    output_dir=str(ROOT / "labs" / "outputs" / "distilbert_sst2"),
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",  # run validation at end of each epoch
    logging_steps=50,
    save_strategy="no",  # skip checkpoint files for this short demo
    report_to="none",  # disable external logging integrations
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
metrics = trainer.evaluate()

print("Training results")
print(f"  Training loss:        {train_result.training_loss:.4f}")
print(f"  Validation accuracy:  {metrics['eval_accuracy']:.4f} ({metrics['eval_accuracy']:.1%})")
print(f"  Validation loss:      {metrics['eval_loss']:.4f}")


### Step 5: Predict five validation sentences

**What this does:** Runs the fine-tuned model on five held-out reviews and prints a table with true label, prediction, and confidence.

**Why we do this:** Accuracy on five examples is easy to inspect manually. Compare mistakes with ambiguous wording (sarcasm, mixed sentiment).


**What to look for in the output**

Table with text snippets, true/pred labels, confidence percentages. Most rows should be correct after fine-tuning. Accuracy on these 5 printed at the bottom.


In [ ]:
model.eval()
n_preds = 5
val_rows = raw["validation"].select(range(n_preds))

pred_table = []
for i in range(n_preds):
    text = val_rows[i]["text"]
    true_label = val_rows[i]["label"]
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1)[0]  # convert scores to probabilities
        pred_label = int(probs.argmax().item())
        confidence = float(probs[pred_label].item())

    pred_table.append({
        "text": text[:80] + ("..." if len(text) > 80 else ""),
        "true_label": LABEL_NAMES[true_label],
        "pred_label": LABEL_NAMES[pred_label],
        "confidence": f"{confidence:.1%}",
        "correct": pred_label == true_label,
    })

pred_df = pd.DataFrame(pred_table)
print("Validation predictions (first 5 examples)")
print(pred_df.to_string(index=False))
print(f"\nAccuracy on these 5: {pred_df['correct'].mean():.0%}")
